# 04 — Modelos predictivos: series, segmentación y clasificación

## Objetivo

Este notebook sirve como guion técnico y visual para la exposición. Todas las
comprobaciones usan los artefactos completos del proyecto; las vistas de pocas filas
son únicamente para mostrar el esquema, no son el conjunto usado por el pipeline.

**QUÉ EXPLICAR:** primero diga qué problema resuelve esta etapa, después muestre la
evidencia y termine conectándola con la siguiente capa.

## Preparación

Ejecútelo desde el contenedor `spark` con la raíz `/workspace`. La celda también
encuentra el repositorio cuando el notebook se abre desde la carpeta local.

In [ ]:
from pathlib import Path
import json

# Encontrar la raíz permite usar el mismo notebook en Docker o en el repositorio local.
candidates = [Path.cwd(), Path.cwd().parent, Path('/workspace')]
ROOT = next(
    path for path in candidates
    if (path / 'config' / 'pipeline.yaml').exists()
)
DATA = ROOT / 'data'
EXPORTS = ROOT / 'exports'
print(f'Raíz del proyecto: {ROOT}')

## Pasos

### 1. Evidencia de modelos Spark ML persistidos

Se entrenan GBT para pronóstico, K-Means para segmentar zonas y Random Forest para
clasificar alta demanda. Todos usan semilla fija y separación temporal donde corresponde.

**QUÉ EXPLICAR:** no basta una predicción; se muestran métrica, partición y artefacto.

In [ ]:
model_root = ROOT / 'artifacts' / 'models'
expected = ['time_series_forecast', 'zone_segmentation', 'high_demand_classifier']
for name in expected:
    success = list((model_root / name).rglob('_SUCCESS'))
    print(name, 'persistido' if success else 'FALTA')
    assert success

### 2. Métricas publicadas en Gold

In [ ]:
import pyarrow.dataset as ds

metric_tables = ['model_timeseries_metrics', 'model_segmentation_metrics',
                 'model_classification_metrics']
for name in metric_tables:
    dataset = ds.dataset(DATA / 'gold' / name, format='parquet', partitioning='hive')
    frame = dataset.to_table().to_pandas()
    print(f'\n{name}')
    print(frame.to_string(index=False))

## Comprobaciones

In [ ]:
report = json.loads((EXPORTS / 'verification_report.json').read_text(encoding='utf-8'))
check = next(c for c in report['checks'] if c['name'] == 'three_predictive_models_and_metrics')
assert check['passed'] and check['details']['models'] == 3
print(check['details'])

## Siguiente paso

Muestre cómo cada ejecución y resultado queda registrado en MongoDB y en exportaciones
CSV/JSON para el dashboard de auditoría.